##Installing required library

In this step,
we install the "PyPDF2" library to read and extract text from the course book PDF file.

This is necessary because machine learning models cannot directly process PDF files, so we must first convert the content into plain text format.

In [1]:
!pip install PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 13.0 MB/s eta 0:00:00


Here, we load the course book PDF file and extract the text from it so we can use it for further processing.

In [5]:
import PyPDF2

pdf_path = "Deep_Learning_with_Python_Second_Editio.pdf"

text = ""

with open(pdf_path, "rb") as file:
    reader = PyPDF2.PdfReader(file)

    for page in reader.pages:
        text += page.extract_text()

print(text[:1000])

MANNINGFrançois CholletSECOND EDITIONDeep Learning with Python
Licensed to kari Jyrkkä <kari.jyrkka@oamk.fi>Licensed to kari Jyrkkä <kari.jyrkka@oamk.fi>Deep Learning
with Python
SECOND  EDITION
FRANÇOIS  CHOLLET
MANNING
SHELTER  ISLAND
Licensed to kari Jyrkkä <kari.jyrkka@oamk.fi>For online information and ordering of this and other Manning books, please visit
www.manning.com . The publisher offers discounts on this book when ordered in quantity. 
For more information, please contact
Special Sales Department
Manning Publications Co.
20 Baldwin Road
PO Box 761Shelter Island, NY 11964
Email: orders@manning.com
©2021 by Manning Publications Co. All rights reserved.
No part of this publication may be reproduced, stored in a retrieval system, or transmitted, in 
any form or by means electronic, mechanical, photocopying, or otherwise, without prior written 
permission of the publisher.
Many of the designations used by manufacturers and sellers to distinguish their products are 
claimed as t

I used this step to fix a file name error and make sure the file is correctly loaded.

In [4]:
import os
print(os.listdir())

['.config', 'Deep_Learning_with_Python_Second_Editio.pdf', 'sample_data']


 installed tiktoken to convert text into tokens for the model.

In [6]:
!pip install tiktoken


used the tiktoken library to convert the extracted text into tokens.

First, loaded the GPT-2 tokenizer. Then, we encode the text into tokens using encode().

Finally,  printed the first few tokens and the total number of tokens to understand how the text is processed.

In [7]:
import tiktoken

# Load GPT-2 tokenizer
enc = tiktoken.get_encoding("gpt2")

# Convert text to tokens
tokens = enc.encode(text)

# Show results
print(tokens[:50])   # first 50 tokens
print("Total tokens:", len(tokens))

[10725, 15871, 38848, 16175, 10924, 609, 349, 1616, 23683, 18672, 39219, 29744, 18252, 351, 11361, 198, 26656, 15385, 284, 479, 2743, 449, 2417, 28747, 11033, 1279, 74, 2743, 13, 73, 2417, 74, 4914, 31, 78, 321, 74, 13, 12463, 29, 26656, 15385, 284, 479, 2743, 449, 2417, 28747, 11033, 1279]
Total tokens: 329108


prepared the training data for the model using a sliding window approach.

We set a context length of 100, which means the model will look at 100 tokens as input.

For each position in the token list, we take 100 tokens as input (X) and the next token as the output (y).

This helps the model learn how to predict the next token based on previous tokens.

In [8]:
context_length = 100

X = []
y = []

for i in range(len(tokens) - context_length):
    X.append(tokens[i:i+context_length])
    y.append(tokens[i+context_length])

print("Input shape:", len(X))
print("Output shape:", len(y))

Input shape: 329008
Output shape: 329008


here we split the dataset into training and validation sets using train_test_split.

We use 80% of the data for training and 20% for validation.

The training data is used to train the model, while the validation data is used to evaluate its performance.
AND  printed the size of both datasets.

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train size:", len(X_train))
print("Validation size:", len(X_val))

Train size: 263206
Validation size: 65802


We build a neural network model where the embedding layer processes tokens and dense layers learn patterns to predict the next token.

In this step, we build a simple neural network model for predicting the next token.

The embedding layer converts tokens (numbers) into meaningful vector representations, so the model can understand relationships between words.

Then, dense layers learn patterns from these vectors and try to predict the next token.

However, this model is very simple and not suitable for a real Large Language Model (LLM). Dense layers cannot understand long context or relationships in text.

Modern LLMs use advanced architectures like Transformers, which are much better at handling language.

In [10]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models

# Convert to numpy arrays
X_train = np.array(X_train)
X_val = np.array(X_val)
y_train = np.array(y_train)
y_val = np.array(y_val)

vocab_size = enc.n_vocab  # from tokenizer

model = models.Sequential([
    layers.Embedding(input_dim=vocab_size, output_dim=64, input_length=100),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(vocab_size, activation='softmax')
])

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

We train the model and evaluate it using validation data.

 FOR UNDERSTANDING

1. X_train → input
2. y_train → correct answer
3. model tries → learn pattern

4. epochs=1 → runs once
5. batch_size=32 → learns in small groups

In [11]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=1,
    batch_size=32
)

8226/8226 ━━━━━━━━━━━━━━━━━━━━ 1466s 178ms/step - accuracy: 0.1796 - loss: 6.0175 - val_accuracy: 0.2221 - val_loss: 5.5734


used the trained model to generate new tokens.

The model looks at the last 100 tokens and predicts the next token.

This process is repeated to generate multiple new tokens.

Finally, these tokens can be converted back into text.

In [23]:
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

def sample_with_temperature(pred, temperature=1.0):
    pred = np.log(pred[0] + 1e-9) / temperature
    pred = np.exp(pred)
    pred = pred / np.sum(pred)
    return np.random.choice(len(pred), p=pred)

def makePredictions(tokens, T):
    tokens = list(tokens)
    outputs = []

    for i in range(T):
        # take last 100 tokens (same as training context)
        input_seq = tokens[-100:]

        # pad if needed
        padded = pad_sequences([input_seq], maxlen=100, padding="pre")

        # predict next token
        pred = model.predict(padded, verbose=0)

        next_token = sample_with_temperature(pred)

        tokens.append(next_token)
        outputs.append(next_token)

    return outputs

here we generate new tokens using the model and convert them back into readable text.

First, we predict 20 new tokens. Then, we decode these tokens into text and print the result.

In [24]:
predictedTokens = makePredictions(tokens, 20)
decoded = enc.decode(predictedTokens)

print(decoded)

 dive YOUR bits per uses 100 cl the’t derivatives all fall visualization your EmbtheAs


The model has learned basic token patterns, but lacks sufficient training to generate meaningful text.

This project was developed with the help of ChatGPT for guidance, explanations, and code structure.

The following prompts were used during the development process:


Reading and extracting text from PDF files using Python.

Converting text into tokens using the tiktoken library.

Preparing training data using a context window approach.

Splitting dataset into training and validation sets.

Building a neural network model using TensorFlow/Keras.

Understanding the role of embedding layers in NLP.

Training the model and evaluating performance.

Generating predictions from the trained model.



##ChatGPT was used to support understanding and implementation, while the final code and explanations were written and verified independently
